# Agentic MDS — IMDS inbox agent

Loads secrets from the Colab 🔑 panel (never from this notebook). **Downloads** `imds_decisions.py` and `imds_agent_v2.py` from GitHub. Do not paste those files into a `%%writefile` cell — Colab treats backslash-open-paren as LaTeX and breaks regex strings.

Then run `--self-test` (no IMDS login), then optionally the live agent.

Required secrets: `IMDS_USERNAME`, `IMDS_PASSWORD`, `OTP_SECRET` (authenticator TOTP seed, **not** a Gmail app password).

Optional: `IMDS_CONTACT_NAME`, `RECIPIENT_COMPANY_IDS`, `IMDS_AUTO_FORWARD` (default off), `IMDS_KILL_SWITCH`, `NUM_ITERATIONS`.
Create `imds_output/KILL` to stop a run.


In [ ]:
%pip install playwright openpyxl nest_asyncio pyotp
!playwright install --with-deps chromium


In [ ]:
import os

# Load secrets from Colab Secrets (🔑 left sidebar). Never commit passwords.
# Required: IMDS_USERNAME, IMDS_PASSWORD, OTP_SECRET
# OTP_SECRET is the authenticator TOTP seed (base32), NOT a Gmail app password.
try:
    from google.colab import userdata
    for _key in (
        "IMDS_USERNAME",
        "IMDS_PASSWORD",
        "OTP_SECRET",
        "IMDS_CONTACT_NAME",
        "RECIPIENT_COMPANY_IDS",
        "IMDS_AUTO_ACCEPT",
        "IMDS_AUTO_REJECT",
        "IMDS_AUTO_FORWARD",
        "IMDS_KILL_SWITCH",
        "NUM_ITERATIONS",
        "IMDS_REQUIRE_PARTS_MARKING",
    ):
        try:
            os.environ[_key] = userdata.get(_key)
        except Exception:
            pass
except ImportError:
    pass

os.environ.setdefault("IMDS_INBOX_URL", "https://www.mdsystem.com/imdsnt/faces/sentReceivedSearch")
print("Secrets loaded from Colab/env. IMDS_USERNAME set:" , bool(os.getenv("IMDS_USERNAME")))


In [ ]:
# Download scripts from GitHub. Do not use %%writefile for these files:
# Colab/IPython treats backslash-open-paren as LaTeX and splits regex strings.
from pathlib import Path
from urllib.request import Request, urlopen

CANDIDATES = [
    "https://raw.githubusercontent.com/rockyforever8-sys/Agentic-MDS/cursor/colab-regex-syntax-07ca",
    "https://raw.githubusercontent.com/rockyforever8-sys/Agentic-MDS/main",
]


def download(name: str) -> str:
    last_err = None
    for base in CANDIDATES:
        url = f"{base}/{name}"
        try:
            with urlopen(Request(url, headers={"User-Agent": "colab"})) as resp:
                data = resp.read()
            Path(name).write_bytes(data)
            return f"{url} ({len(data)} bytes)"
        except Exception as exc:
            last_err = exc
    raise RuntimeError(f"Could not download {name}: {last_err}")


for name in ("imds_decisions.py", "imds_agent_v2.py"):
    print(download(name))

import py_compile
py_compile.compile("imds_decisions.py", doraise=True)
py_compile.compile("imds_agent_v2.py", doraise=True)
print("compile OK")


In [ ]:
# No IMDS login. Verifies green/amber/red scoring and writes fixture Excel + JSONL.
!python imds_agent_v2.py --self-test


In [ ]:
# Live IMDS run. Skip this cell until --self-test is green and secrets are set.
# Kill switch: IMDS_KILL_SWITCH=1 or create imds_output/KILL
!python imds_agent_v2.py
